In [1]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults  ## search tool  used for internet search
from langchain import hub
from langsmith import Client
from langchain.tools import tool


In [2]:
from langchain.agents import create_react_agent, AgentExecutor   ## react ----> Reasoning and action == ReAct this is langchain agent to create our agent


In [4]:
## load env variable

# os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
WEATHERSTACK_API_KEY=os.getenv("WEATHERSTACK_API_KEY")



In [5]:
search_tool= TavilySearchResults(max_results=2)

/var/folders/p4/697jx8ns5d3_5hf0yn67fywh0000gn/T/ipykernel_74915/4218638880.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  search_tool= TavilySearchResults(max_results=2)


In [ ]:
@tool         ##if i want to use custom function as tool then i have to use decorator (@) now it become custom tool
def get_weather_data(city:str) -> str :
        """
        Fetch current weather information for a city.
        """

        url=(
                f"https://api.weatherstack.com/current?"
                f"access_key={WEATHERSTACK_API_KEY}&query={city}"
        )
        print(url)

        response = requests.get(url)

        data = response.json()

        if "current" not in data:
                return f"Could not fetch weather data for {city}"
        
        print(data['current'])

        return (
                f"City:{city}\n"
                f"Temperature: {data['current']['temperature']}°C\n"  ##option+shift+8 for degree sign
                f"Weather: {data['current']['weather_descriptions'][0]}\n"
                f"Humidity: {data['current']['humidity']}%"
        )

In [14]:
result = get_weather_data("delhi")
result

##"https://api.weatherstack.com/current?access_key=9b5d39aa8ad1676a188f827425ae0c71&query=delhi"

'Could not fetch weather data for delhi'

In [78]:
result = search_tool.invoke("What is the current score of India in cricket today match")
result

[{'title': "Live Cricket Score, Today's Cricket Match Live Scorecard & ...",
  'url': 'https://indianexpress.com/section/sports/cricket/live-score',
  'content': "265/10 (48.1)\n\nVS\n\nSri Lanka\n\n176/10 (37.0)\n\nSri Lanka\n\nENG vs SL 1st ODI Result: England beat Sri Lanka by 89 runs\n\nView Scorecard\n\nTue 22 Sep '26 9:30 AM IST\n\nResult\n\nOne-off T20I  Sano\n\nJapan\n\nJapan\n\n30/3 (5.0)\n\nVS\n\nIndia\n\n32/4 (5.0)\n\nIndia\n\nJPN vs IND One-off T20I Result: India beat Japan by 2 runs\n\nView Scorecard\n\nSun 20 Sep '26 1:00 PM IST\n\nResult\n\n3rd ODI  Harare\n\nZimbabwe\n\nZimbabwe\n\n271/8 (50.0)\n\nVS\n\nAustralia\n\n272/9 (49.5)\n\nAustralia\n\nZIM vs AUS 3rd ODI Result: Australia beat Zimbabwe by 1 wicket\n\nView Scorecard\n\nSun 20 Sep '26 6:30 AM IST\n\nResult\n\n2nd T20I  Sano\n\nJapan\n\nJapan\n\nVS\n\nHong Kong\n\nHong Kong\n\nJPN vs HK 2nd T20I Result: Match Abandoned\n\nView Scorecard\n\nSat 19 Sep '26 7:00 PM IST\n\nResult\n\n3rd T20I  Manchester\n\nEngland\n\n

In [6]:
## single agent system now

## now initailize llm

llm= ChatOpenAI(
        model="gpt-3.5-turbo",
        temperature=0,
        api_key=OPENAI_API_KEY
)

In [98]:
llm_1=ChatGroq(
        model="openai/gpt-oss-120b",
        api_key=GROQ_API_KEY
)

In [99]:
## to check llm is working or not we can invoke it

response= llm_1.invoke("what is your last trained year")
response

AIMessage(content='My training data includes information up through **June\u202f2024**.', additional_kwargs={'reasoning_content': 'The user asks: "what is your last trained year". They want to know the cutoff year of training data. According to system prompt: "knowledge cutoff: 2024-06". So answer: My training data goes up to June 2024. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 77, 'total_tokens': 156, 'completion_time': 0.164300315, 'completion_tokens_details': {'reasoning_tokens': 56}, 'prompt_time': 0.003097418, 'prompt_tokens_details': None, 'queue_time': 0.34807321, 'total_time': 0.167397733}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4200b3f836', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a0e44d-59e1-7ff0-afcd-5eba7d5944ac-0', usage_metadata={'input_tokens': 77, 'output_tokens': 79, 'total_tokens': 156})

In [107]:
## to download prompt from langchain hub (need langsmith) or you can write it manually


client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [103]:
tools=[search_tool, get_weather_data]

In [108]:
## now we create the agent

agent = create_react_agent(
        llm=llm_1,
        tools=tools,
        prompt=prompt
)

In [110]:
## agent executor

agent_executor= AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=True ,   ## to see the log of plans and all (to able to see execution process)
        handle_parsing_errors=True    
)

In [113]:
## ro run

response = agent_executor.invoke({
        "input": (
                "Find the capital of India"
                "and then find its current weather."
        )
})



> Entering new AgentExecutor chain...
Question: Find the capital of India and then find its current weather.  
Thought: I need to determine the capital city of India, then retrieve the current weather for that city.  
Action: tavily_search_results_json  
Action Input: "capital of India"  
[{'title': 'New Delhi', 'url': 'https://en.wikipedia.org/wiki/New_Delhi', 'content': 'Edit links\n\n Article\n Talk\n\n Read\n View source\n View history\n\nTools\n\nActions\n\n Read\n View source\n View history\n\nGeneral\n\n What links here\n Related changes\n Upload file\n Permanent link\n Page information\n Cite this page\n Get shortened URL\n Switch to legacy parser\n\nPrint/export\n\n Download as PDF\n Printable version\n\nIn other projects\n\n Abstract Wikipedia\n Wikimedia Commons\n Wikinews\n Wikiquote\n Wikivoyage\n Wikidata item\n\nAppearance\n\nCoordinates: 28°36′50″N 77°12′32″E\ufeff / \ufeff28.6139°N 77.2089°E\ufeff / 28.6139; 77.2089\n\nPage semi-protected\n\nFrom Wikipedia, the free 

KeyboardInterrupt: 

In [90]:
print(response["output"])

As of September 2026, there is no war between Iran and the United States. The two countries remain in a state of strained diplomatic relations, marked by occasional diplomatic talks, economic sanctions, and regional security concerns, but they have not engaged in an armed conflict that would be described as a war. Recent news has focused on issues such as nuclear negotiations, regional proxy activities, and occasional incidents involving naval vessels or aircraft, all of which have been managed through diplomatic channels, sanctions, and international mediation rather than open warfare.


                llm
                 |

user --------> agent --------->output
|
tavily
